<a href="https://colab.research.google.com/github/shridharch/support-ticket-triage-langchain/blob/main/notebooks/ticket_triage_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **#Support Ticket Triage with** **LangChain** **bold text**

### **"Setup: install packages"**

In [1]:
  !pip install -q langchain langchain-groq langgraph pydantic

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "ticket-triage"

In [3]:
!pip install -q langchain langchain-google-genai langgraph pydantic

In [4]:
from groq import Groq
for m in Groq().models.list().data:
    print(m.id)

whisper-large-v3-turbo
qwen/qwen3.8-27b
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
openai/gpt-oss-20b


## **"Load the model"**

In [5]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
print(llm.invoke("Say hello in one sentence.").text)

Hello!


## **"Build the chain"**

In [6]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Ticket(BaseModel):
    category: str = Field(description="One of: billing, bug, feature_request, other")
    urgency: str = Field(description="One of: low, medium, high")
    summary: str = Field(description="One-sentence summary of the issue")

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You triage customer support emails.\n"
     "Categories: billing (charges, refunds, invoices), bug (something broken), "
     "feature_request (asking for new functionality), other.\n"
     "Urgency: high = outage, money lost, or angry customer; "
     "medium = blocked but has workaround; low = general question."),
    ("human", "{email}"),
])

chain = prompt | llm.with_structured_output(Ticket)

print(chain.invoke({"email": "My card was charged twice, fix this ASAP!"}))

category='billing' urgency='high' summary='Card charged twice'


## **"Test on 10 emails"**

In [7]:
import time

emails = [
    "My card was charged twice, fix this ASAP!",
    "The app crashes every time I click Export.",
    "Could you add dark mode? It would be great.",
    "Site is down for all our customers!!",
    "How do I change my profile picture?",
    "I was billed $50 but my plan is $20.",
    "Login button doesn't work on Safari, using Chrome instead for now.",
    "Love the product, just saying thanks.",
    "Please add CSV export for reports.",
    "Refund me now or I'm cancelling and posting reviews everywhere!",
]

lines = []
for e in emails:
    try:
        r = chain.invoke({"email": e})
        lines.append(f"{r.category:16} {r.urgency:7} | {e}")
    except Exception as ex:
        lines.append(f"{'FAILED':16} {'-':7} | {e}  <- {type(ex).__name__}")
    time.sleep(3)

print("\n".join(lines))

billing          high    | My card was charged twice, fix this ASAP!
bug              high    | The app crashes every time I click Export.
feature_request  low     | Could you add dark mode? It would be great.
bug              high    | Site is down for all our customers!!
other            low     | How do I change my profile picture?
billing          high    | I was billed $50 but my plan is $20.
bug              medium  | Login button doesn't work on Safari, using Chrome instead for now.
other            low     | Love the product, just saying thanks.
feature_request  low     | Please add CSV export for reports.
billing          high    | Refund me now or I'm cancelling and posting reviews everywhere!


# **Next: Stage 2 (tools)**

### Define two **tools**

In [8]:
from typing import Literal
from langchain_core.tools import tool

@tool
def create_ticket(
    category: Literal["billing", "bug", "feature_request", "other"],
    urgency: Literal["low", "medium", "high"],
    summary: str,
) -> str:
    """Save a support ticket to the database."""
    print(f"[create_ticket] {category} | {urgency} | {summary}")
    return "Ticket #123 created"

@tool
def notify_team(message: str) -> str:
    """Send an urgent alert to the support team. Use only for high-urgency tickets."""
    print(f"[notify_team] {message}")
    return "Team notified"

**let the model choose tools**

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

llm_with_tools = llm.bind_tools([create_ticket, notify_team])
tools = {"create_ticket": create_ticket, "notify_team": notify_team}

system = SystemMessage(
    "You handle support emails.\n"
    "Step 1: call create_ticket with category (billing, bug, feature_request, other) "
    "and urgency (low, medium, high).\n"
    "Step 2: if urgency is high (outage, money lost, or angry customer), you MUST also "
    "call notify_team.\n"
    "Do not skip notify_team for high urgency."
)

def handle(email, max_rounds=3):
    messages = [system, HumanMessage(email)]
    print("EMAIL:", email)
    for _ in range(max_rounds):
        ai = llm_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            break
        for call in ai.tool_calls:
            result = tools[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    print("-" * 40)

## `Test it`

In [10]:
handle("Site is down for all our customers!!")
handle("How do I change my profile picture?")

EMAIL: Site is down for all our customers!!
[create_ticket] bug | high | Site is down for all customers
[notify_team] Urgent: Site is down for all customers. Immediate attention required.
----------------------------------------
EMAIL: How do I change my profile picture?
[create_ticket] other | low | User asks how to change profile picture
----------------------------------------
